### RaschPy model selection worked example

RaschPy provides three model-selection tools for choosing the simplest model structure that adequately fits the data:

1. **RSM vs PCM** (`model_selection()`, on either model): is a single shared threshold structure (RSM) adequate, or are independent per-item thresholds (PCM) required to explain the data?
2. **MFRM rater parameterisation** (`model_selection()`, on `MFRM`): of the eight rater representations (global, items, thresholds, bivector, matrix, centrality, pseudo_halo, bistretch), achieves the optimal balance between explanation and complexity?
3. **MFRM "mixed" model** (`per_rater_model_selection()`): rather than forcing every rater into the same parameterisation, assign each rater individually the model that achieves the optimal balance between explanation and complexity.

All three support likelihood-ratio, AIC, and BIC criteria. This notebook simulates data under a known structure in each case and checks whether these criteria correctly recover the generating structure(s).

Import `raschpy`:

In [1]:
import raschpy as rp

#### RSM vs PCM

Simulate data generated under RSM and check whether `model_selection()` (available on either `RSM` or `PCM`) correctly prefers RSM:

In [2]:
sim_rsm = rp.RSM_Sim(no_of_items=15, no_of_persons=1500, max_score=4, seed=1)
rsm = rp.RSM(sim_rsm)
rsm.calibrate()
rsm.model_selection(test='AIC')
rsm.model_comparison_rsm_pcm_aic_summary

PCM AIC                          205078.668694
RSM AIC                           204541.85378
Relative likelihood                        1.0
Test level                            0.000024
delta                                      0.1
p-value (close fit)                        1.0
p-value (close fit, per item)              1.0
Delta-hat                                  0.0
Units                                    width
Preferred                                  RSM
Name: RSM vs PCM AIC comparison (composite), dtype: object

Simulate data genuinely generated under PCM and check that `model_selection()` correctly prefers PCM:

In [3]:
sim_pcm = rp.PCM_Sim(no_of_items=15, no_of_persons=2000, max_score_vector=[4] * 15, category_base=3, seed=1)
pcm = rp.PCM(sim_pcm)
pcm.calibrate()
pcm.model_selection(test='AIC')
pcm.model_comparison_rsm_pcm_aic_summary

PCM AIC                          193121.215232
RSM AIC                          260348.649894
Relative likelihood                        0.0
Test level                            0.000024
delta                                      0.1
p-value (close fit)                        0.0
p-value (close fit, per item)              0.0
Delta-hat                              0.42004
Units                                    width
Preferred                                  PCM
Name: RSM vs PCM AIC comparison (composite), dtype: object

`test='LR'` and `test='BIC'` are also available — results are stored as `model_comparison_rsm_pcm_{lr,aic,bic}_summary` respectively (plus `_preferred`, and for AIC with `aic_relative_likelihood=True`, the relative likelihood of RSM and the level of the test, `_aic_relative_likelihood` and `_aic_level`).

#### MFRM rater-parameterisation model selection

Simulate data genuinely generated under the `items` rater parameterisation (a separate effect per rater x item combination), using `MFRM_Sim_Items`, and check that `model_selection()` (which by default calibrates all eight parameterisations and ranks them) correctly identifies it:

In [4]:
sim_mfrm = rp.MFRM_Sim_Items(no_of_items=10, no_of_persons=500, no_of_facet_elements=8, max_score=4,
                             item_range=3, facet_range=2, seed=1)
mfrm = rp.MFRM(sim_mfrm)
mfrm.calibrate_global()   # any one calibration is enough to instantiate the model
mfrm.model_selection(test='AIC')
mfrm.model_comparison_mfrm_aic_summary.round(2)

,CL,k,Penalty,AIC,ΔAIC
Model,,,,,
items,-204257.28,82.0,613.91,409742.38,-22784.92
bivector,-204145.21,103.0,853.36,409997.13,-22530.17
matrix,-202771.04,292.0,3101.84,411745.76,-20781.54
bistretch,-215044.33,33.0,212.46,430513.58,-2013.72
pseudo_halo,-215282.81,26.0,150.03,430865.67,-1661.63
centrality,-215798.68,26.0,159.46,431916.28,-611.02
thresholds,-215660.93,40.0,344.97,432011.80,-515.50
global,-216168.18,19.0,95.47,432527.30,0.00


In [5]:
mfrm.model_comparison_mfrm_aic_preferred

'items'

By default, `model_selection()` calibrates and compares all eight rater parameterisations. Pass `candidate_models=` to restrict the comparison to a selected subset. For example, to test only whether `pseudo_halo`'s 2-parameter item-stretch restriction is needed over the full `items` parameterisation (the model this data was actually generated under, so `items` should win convincingly):

In [6]:
mfrm.model_selection(test='LR', candidate_models=['items', 'pseudo_halo'])
mfrm.model_comparison_mfrm_lr_summary

,LR,df,Scale,df (Satterthwaite),p-value,Departure,p-value (close fit),Effect,Preferred
Comparison,,,,,,,,,
pseudo_halo vs items,22051.056804,56,8.972101,50.522291,0.0,0.201519,0.015506,1.199398,items


The `candidate_models=` argument restricts the set of models under consideration to those passed as a list, so can be useful when you are interested in a specific subset of the model family, for example the 'stretch' models. Restricting to just the three stretch models also demonstrates `aic_relative_likelihood`'s baseline logic when the standard beseline `global` is not in the set: the baseline becomes whichever model has the fewest free parameters instead -- here `centrality` and `pseudo_halo` are tied (both 2-parameter restrictions), so the tie is broken by lowest AIC:

In [7]:
mfrm.model_selection(test='AIC', candidate_models=['centrality', 'pseudo_halo', 'bistretch'])
mfrm.model_comparison_mfrm_aic_summary

/Users/markelliott/Documents/RaschPy-dev/raschpy/mfrm.py:74: UserWarning: model_selection(): the default tolerance delta=0.2 is in median category widths, and no candidate has shared thresholds to take the width from; steps adding item or threshold effects therefore have no tolerance. To use one, set delta in logits with delta_units='logits'. The stretch tolerance (stretch_delta) still applies.
  return func(self, *args, **kwargs)


,CL,k,Penalty,AIC,ΔAIC
Model,,,,,
bistretch,-215044.329902,33.0,212.459197,430513.578199,-352.094260
pseudo_halo,-215282.809619,26.0,150.026611,430865.672459,0.000000
centrality,-215798.677084,26.0,159.462295,431916.278760,1050.606301


#### MFRM mixed rater model

Real rating designs rarely have every rater behaving with the same complexity: some raters are simply more or less severe (`'global'`), while others need a more complex representation. Rather than forcing one representation on every rater, `per_rater_model_selection()` assigns each rater its own. Each candidate representation for a rater is a joint fit of all raters (`calibrate(model='mixed')`), with the other raters on their current representations; the candidates are compared by AIC, and a more complex representation than `'global'` is kept only if `'global'`'s relative likelihood, exp(−ΔAIC/2), is at or below `alpha=0.05` (`aic_relative_likelihood=True`: a likelihood ratio test at a level below `alpha`, so more conservative than `test='LR'`) and its departure from the simpler representation is larger than a tolerance (`delta`, 0.1 median category widths by default). The default `scheme='iterate'` updates one rater at a time until a pass changes nothing. `test='LR'` climbs the same ladder with likelihood ratio tests; `min_effect=` is available with `estimation='pair-cpat'` only.

With `anchors=[...]`, each rater's representation is read against the anchors, whose average defines the zero point. The anchors start on `'global'` and are selected like the other raters (`force_global_anchors=True` holds them there); whether they agree with each other is tested by `check_anchor_homogeneity()` (see the Anchor calibration example).

Simulate 8 raters: 4 behave as `'global'` raters with zero global effect, and 4 show a per-item "halo" pattern (more lenient on some items, more severe on others), i.e. `'items'` behaviour. Anchor to the four raters with zero global effect:

In [8]:
max_score = 4
no_of_items = 8

flat = {f'Item_{i + 1}': [0] * max_score for i in range(no_of_items)}
halo = {f'Item_{i + 1}': [(((no_of_items - 1) / 2) - i) / 3] * max_score for i in range(no_of_items)}

sim_mixed = rp.MFRM_Sim_Matrix(no_of_items=no_of_items, no_of_persons=2000, no_of_facet_elements=8, max_score=max_score, person_sd=4,
                               manual_raters={'Rater_1': flat, 'Rater_2': flat, 'Rater_3': flat, 'Rater_4': flat,
                                              'Rater_5': halo, 'Rater_6': halo, 'Rater_7': halo, 'Rater_8': halo},
                               seed=1)

generating_models = {'Rater_1': 'global', 'Rater_2': 'global', 'Rater_3': 'global', 'Rater_4': 'global',
                     'Rater_5': 'items', 'Rater_6': 'items', 'Rater_7': 'items', 'Rater_8': 'items'}

mfrm_mixed = rp.MFRM(sim_mixed)
anchors = ['Rater_1', 'Rater_2', 'Rater_3', 'Rater_4']
mfrm_mixed.per_rater_model_selection(anchors=anchors)
mfrm_mixed.per_rater_model_selection_table[['Representation', 'AIC global', 'AIC items', 'AIC thresholds', 'AIC bivector']].round(1)

,Representation,AIC global,AIC items,AIC thresholds,AIC bivector
rater,,,,,
Rater_1,global,703524.27159,703583.715552,703571.779585,703630.13036
Rater_2,global,703524.27159,703544.842298,703553.504754,703575.088891
Rater_3,global,703524.27159,703572.794333,703573.860099,703622.935581
Rater_4,global,703524.27159,703554.586549,703511.02635,703541.001335
Rater_5,items,718521.045157,703524.27159,718113.844379,703525.597939
Rater_6,items,719729.863349,703524.27159,719449.599424,703578.33742
Rater_7,items,717865.616476,703524.27159,717641.667424,703568.358261
Rater_8,items,717932.305464,703524.27159,717545.798237,703495.883727


`rater_models` gives each rater's representation, and `per_rater_model_selection_counts` how many raters landed on each:

In [9]:
mfrm_mixed.rater_models

Rater_1    global
Rater_2    global
Rater_3    global
Rater_4    global
Rater_5     items
Rater_6     items
Rater_7     items
Rater_8     items
Name: Representation, dtype: str

In [10]:
mfrm_mixed.per_rater_model_selection_counts

Representation
global    4
items     4
Name: count, dtype: int64

Compare with the generating representation of each rater:

In [11]:
import pandas as pd

comparison = pd.DataFrame({'Generating': pd.Series(generating_models),
                           'Selected': mfrm_mixed.rater_models})
comparison['Correct'] = (comparison['Generating'] == comparison['Selected'])
comparison

,Generating,Selected,Correct
Rater_1,global,global,True
Rater_2,global,global,True
Rater_3,global,global,True
Rater_4,global,global,True
Rater_5,items,items,True
Rater_6,items,items,True
Rater_7,items,items,True
Rater_8,items,items,True


All eight raters are assigned their generating representation. For two of them a larger representation has the lower AIC: Rater_8's `'bivector'` (703,495.9 against 703,524.3 for its generating `'items'`) and Rater_4's `'thresholds'` (703,511.0 against 703,524.3 for `'global'`). With `test='AIC'`, a step up the ladder is kept only if it passes both the relative likelihood check and the tolerance test, and neither step was kept.

With every rater assigned a representation, the statistics tables of the mixed model are produced as for any single representation, with `model='mixed'`. The fit is already in its anchors' frame, so passing the same anchors (or none) gives the fit as it is:

In [12]:
mfrm_mixed.rater_stats_df(model='mixed', anchors=anchors, full=True, marginal=False)

Item_1, Threshold 1                      Item_1, Threshold 2         \
                   Estimate     SE   2.5%  97.5%            Estimate     SE   
Rater_1              -0.017  0.012 -0.042  0.008              -0.017  0.012   
Rater_2               0.003  0.013 -0.022  0.027               0.003  0.013   
Rater_3               0.005  0.013 -0.020  0.030               0.005  0.013   
Rater_4               0.009  0.012 -0.015  0.033               0.009  0.012   
Rater_5               1.243  0.046  1.152  1.333               1.243  0.046   
Rater_6               1.211  0.048  1.116  1.306               1.211  0.048   
Rater_7               1.133  0.046  1.043  1.223               1.133  0.046   
Rater_8               1.193  0.047  1.101  1.285               1.193  0.047   

                      Item_1, Threshold 3         ... Item_8, Threshold 4  \
          2.5%  97.5%            Estimate     SE  ...            Estimate   
Rater_1 -0.042  0.008              -0.017  0.012  ...              -0.017   
Rater_2 -0.022  0.027               0.003  0.013  ...               0.003   
Rater_3 -0.020  0.030               0.005  0.013  ...               0.005   
Rater_4 -0.015  0.033               0.009  0.012  ...               0.009   
Rater_5  1.152  1.333               1.243  0.046  ...              -1.163   
Rater_6  1.116  1.306               1.211  0.048  ...              -1.193   
Rater_7  1.043  1.223               1.133  0.046  ...              -1.154   
Rater_8  1.101  1.285               1.193  0.047  ...              -1.129   

                             Rater model Overall statistics                   \
            SE   2.5%  97.5%       Model              Count Infit MS Infit Z   
Rater_1  0.012 -0.042  0.008      global            16000.0    1.001   0.089   
Rater_2  0.013 -0.022  0.027      global            16000.0    0.957  -3.038   
Rater_3  0.013 -0.020  0.030      global            16000.0    0.975  -1.784   
Rater_4  0.012 -0.015  0.033      global            16000.0    1.003   0.193   
Rater_5  0.045 -1.251 -1.076       items            16000.0    0.975  -1.755   
Rater_6  0.043 -1.278 -1.108       items            16000.0    0.985  -1.076   
Rater_7  0.043 -1.239 -1.069       items            16000.0    0.970  -2.113   
Rater_8  0.043 -1.214 -1.044       items            16000.0    0.967  -2.300   

                            
        Outfit MS Outfit Z  
Rater_1     0.870   -2.124  
Rater_2     0.802   -3.200  
Rater_3     0.885   -1.863  
Rater_4     0.862   -2.212  
Rater_5     0.830   -2.508  
Rater_6     0.871   -1.863  
Rater_7     0.838   -2.410  
Rater_8     0.825   -2.636  

[8 rows x 134 columns]

`per_rater_model_selection()` also takes `candidate_models=`, restricting the representations each rater can be assigned: the AIC choice is taken over the models passed, and `aic_relative_likelihood` compares the winner with whichever of them has the fewest parameters. Restricting to `['global', 'items', 'thresholds', 'bivector']` leaves out `'matrix'` and the three stretch models, which is useful when they are not of interest for a given design. Since these data were generated under `'global'` and `'items'` only, the assignment should not change:

In [13]:
mfrm_mixed.per_rater_model_selection(anchors=anchors,
                                      candidate_models=['global', 'items', 'thresholds', 'bivector'])
mfrm_mixed.rater_models

Rater_1    global
Rater_2    global
Rater_3    global
Rater_4    global
Rater_5     items
Rater_6     items
Rater_7     items
Rater_8     items
Name: Representation, dtype: str